In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import glob
import os
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import (
    balanced_accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.api import VAR

warnings.filterwarnings("ignore")

TRAINING_PAIRS = ["EURUSD", "GBPUSD", "USDJPY", "AUDUSD", "USDCHF"]
TRAIN_RATIO = 0.80
LOOKBACK_MONTHS = 12
VAR_MAX_FEATS = 6
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and ROOT.parent.exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
MACRO_DIR = DATA_DIR / "processed" / "macro"
NEWS_DIR = MACRO_DIR / "news"
OHLCV_DIR = DATA_DIR / "processed" / "ohlcv"

print(f"ROOT: {ROOT.resolve()}")
print(f"MACRO_DIR exists: {MACRO_DIR.exists()} -> {MACRO_DIR}")
print(f"NEWS_DIR exists: {NEWS_DIR.exists()} -> {NEWS_DIR}")
print(f"OHLCV_DIR exists: {OHLCV_DIR.exists()} -> {OHLCV_DIR}")

ROOT: D:\SCRIPTS\FX-AlphaLab\notebooks
MACRO_DIR exists: True -> D:\SCRIPTS\FX-AlphaLab\notebooks\data\processed\macro
NEWS_DIR exists: False -> D:\SCRIPTS\FX-AlphaLab\notebooks\data\processed\macro\news
OHLCV_DIR exists: False -> D:\SCRIPTS\FX-AlphaLab\notebooks\data\processed\ohlcv


In [2]:
probe = Path.cwd().resolve()
resolved_root = None
for candidate in [probe, *probe.parents]:
    if (candidate / "pyproject.toml").exists():
        resolved_root = candidate
        break
if resolved_root is None:
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

ROOT = resolved_root
DATA_DIR = ROOT / "data"
MACRO_DIR = DATA_DIR / "processed" / "macro"
NEWS_DIR = MACRO_DIR / "news"
OHLCV_DIR = DATA_DIR / "processed" / "ohlcv"

print(f"Adjusted ROOT: {ROOT}")
print(f"MACRO_DIR exists: {MACRO_DIR.exists()} -> {MACRO_DIR}")
print(f"NEWS_DIR exists: {NEWS_DIR.exists()} -> {NEWS_DIR}")
print(f"OHLCV_DIR exists: {OHLCV_DIR.exists()} -> {OHLCV_DIR}")

Adjusted ROOT: D:\SCRIPTS\FX-AlphaLab
MACRO_DIR exists: True -> D:\SCRIPTS\FX-AlphaLab\data\processed\macro
NEWS_DIR exists: True -> D:\SCRIPTS\FX-AlphaLab\data\processed\macro\news
OHLCV_DIR exists: True -> D:\SCRIPTS\FX-AlphaLab\data\processed\ohlcv


In [3]:
macro_csv_files = sorted(MACRO_DIR.glob("macro_*.csv"))
individual_files = [p for p in macro_csv_files if not p.name.startswith("macro_all_")]
macro_all_files = [p for p in macro_csv_files if p.name.startswith("macro_all_")]

if not individual_files and not macro_all_files:
    raise FileNotFoundError(f"No macro CSV files found in {MACRO_DIR}")

largest_macro_all = max(macro_all_files, key=lambda p: p.stat().st_size) if macro_all_files else None


def _read_macro_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}

    ts_col = None
    for cand in ["timestamp_utc", "timestamp", "date", "datetime", "time"]:
        if cand in cols:
            ts_col = cols[cand]
            break
    if ts_col is None:
        raise ValueError(f"No timestamp-like column in {path.name}; columns={list(df.columns)}")

    series_col = None
    for cand in ["series_id", "series", "indicator", "name"]:
        if cand in cols:
            series_col = cols[cand]
            break
    if series_col is None:
        raise ValueError(f"No series-like column in {path.name}; columns={list(df.columns)}")

    value_col = None
    for cand in ["value", "close", "last", "val"]:
        if cand in cols:
            value_col = cols[cand]
            break
    if value_col is None:
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        numeric_cols = [c for c in numeric_cols if c not in {series_col, ts_col}]
        if not numeric_cols:
            raise ValueError(f"No value-like numeric column in {path.name}; columns={list(df.columns)}")
        value_col = numeric_cols[0]

    out = df[[ts_col, series_col, value_col]].rename(
        columns={ts_col: "timestamp_utc", series_col: "series_id", value_col: "value"}
    )
    out["timestamp_utc"] = pd.to_datetime(out["timestamp_utc"], utc=True, errors="coerce")
    out["series_id"] = out["series_id"].astype(str).str.strip()
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["timestamp_utc", "series_id"])
    return out

frames = []
for file in individual_files:
    try:
        frames.append(_read_macro_csv(file))
    except Exception as e:
        print(f"Skipped {file.name}: {e}")

if largest_macro_all is not None:
    try:
        macro_all_df = _read_macro_csv(largest_macro_all)
        frames.append(macro_all_df)
        print(f"Included macro_all file: {largest_macro_all.name} ({largest_macro_all.stat().st_size} bytes)")
    except Exception as e:
        print(f"Skipped {largest_macro_all.name}: {e}")

if not frames:
    raise RuntimeError("All macro CSV reads failed.")

macro_long = pd.concat(frames, ignore_index=True)
macro_long = macro_long.drop_duplicates(subset=["series_id", "timestamp_utc"]).sort_values(
    ["series_id", "timestamp_utc"]
)

series_ranges = (
    macro_long.groupby("series_id")["timestamp_utc"]
    .agg(["min", "max", "count"])
    .sort_values("series_id")
)

print(f"Rows after dedupe: {len(macro_long):,}")
print(f"Total series loaded: {series_ranges.shape[0]}")
print("Series names:")
print(series_ranges.index.tolist())
print("Date range per series:")
print(series_ranges)

Included macro_all file: macro_all_2021-01-01_2025-12-31.csv (105705 bytes)
Rows after dedupe: 4,980
Total series loaded: 10
Series names:
['BAMLH0A0HYM2', 'BOPGSTB', 'CPIAUCSL', 'DFF', 'ECB_DFR', 'ECB_MRR', 'NETFI', 'STLFSI4', 'UNRATE', 'VIXCLS']
Date range per series:
                                   min                       max  count
series_id                                                              
BAMLH0A0HYM2 2021-01-04 00:00:00+00:00 2026-02-19 00:00:00+00:00   1343
BOPGSTB      2021-01-01 00:00:00+00:00 2025-12-01 00:00:00+00:00     60
CPIAUCSL     2021-01-01 00:00:00+00:00 2025-12-01 00:00:00+00:00     59
DFF          2021-01-01 00:00:00+00:00 2025-12-31 00:00:00+00:00   1826
ECB_DFR      2022-07-27 00:00:00+00:00 2025-06-11 00:00:00+00:00     18
ECB_MRR      2022-07-27 00:00:00+00:00 2025-06-11 00:00:00+00:00     18
NETFI        2021-01-01 00:00:00+00:00 2025-07-01 00:00:00+00:00     19
STLFSI4      2021-01-01 00:00:00+00:00 2025-12-26 00:00:00+00:00    261
UNRATE   

In [4]:
macro_monthly_long = macro_long.copy()
macro_monthly_long["month_end"] = (
    macro_monthly_long["timestamp_utc"].dt.to_period("M").dt.to_timestamp("M").dt.tz_localize("UTC")
)

macro_monthly = (
    macro_monthly_long.pivot_table(
        index="month_end", columns="series_id", values="value", aggfunc="last"
    )
    .sort_index()
)
macro_monthly = macro_monthly.ffill(limit=6)
macro_monthly.index.name = "timestamp_utc"
macro_monthly = macro_monthly.reset_index()

print(f"macro_monthly shape: {macro_monthly.shape}")
print("macro_monthly columns:")
print(macro_monthly.columns.tolist())
print("month range:", macro_monthly["timestamp_utc"].min(), "->", macro_monthly["timestamp_utc"].max())

macro_monthly shape: (62, 11)
macro_monthly columns:
['timestamp_utc', 'BAMLH0A0HYM2', 'BOPGSTB', 'CPIAUCSL', 'DFF', 'ECB_DFR', 'ECB_MRR', 'NETFI', 'STLFSI4', 'UNRATE', 'VIXCLS']
month range: 2021-01-31 00:00:00+00:00 -> 2026-02-28 00:00:00+00:00


In [5]:
hawk_words = ["inflation", "tighten", "hike", "restrictive", "hawkish", "higher for longer"]
dove_words = ["cut", "easing", "accommodative", "recession", "slowdown", "dovish"]

news_sources = ["fed", "ecb", "boe"]
news_frames = []

if not NEWS_DIR.exists():
    print(f"Warning: news directory missing at {NEWS_DIR}; using empty news feature frame.")
    news_monthly_wide = pd.DataFrame(columns=["timestamp_utc"])
else:
    for src in news_sources:
        src_files = sorted((NEWS_DIR / src).glob("year=*/month=*/news_cleaned.parquet"))
        for path in src_files:
            try:
                ndf = pd.read_parquet(path)
                if ndf.empty:
                    continue
                text_col = None
                for cand in ["headline", "title", "text", "content", "summary"]:
                    if cand in ndf.columns:
                        text_col = cand
                        break
                ts_col = None
                for cand in ["timestamp_utc", "timestamp", "published_at", "date", "datetime"]:
                    if cand in ndf.columns:
                        ts_col = cand
                        break
                if text_col is None or ts_col is None:
                    continue

                ndf = ndf[[ts_col, text_col]].rename(columns={ts_col: "timestamp_utc", text_col: "text"})
                ndf["timestamp_utc"] = pd.to_datetime(ndf["timestamp_utc"], utc=True, errors="coerce")
                ndf = ndf.dropna(subset=["timestamp_utc"])
                ndf["text"] = ndf["text"].fillna("").astype(str).str.lower()
                ndf["month_end"] = ndf["timestamp_utc"].dt.to_period("M").dt.to_timestamp("M").dt.tz_localize("UTC")
                ndf["hawk"] = ndf["text"].apply(lambda s: sum(w in s for w in hawk_words))
                ndf["dove"] = ndf["text"].apply(lambda s: sum(w in s for w in dove_words))
                agg = ndf.groupby("month_end").agg(doc_count=("text", "count"), hawk=("hawk", "sum"), dove=("dove", "sum"))
                agg["tone"] = (agg["hawk"] - agg["dove"]) / (1.0 + agg["hawk"] + agg["dove"])
                agg = agg.reset_index().rename(columns={"month_end": "timestamp_utc"})
                agg["source"] = src.upper()
                news_frames.append(agg)
            except Exception as e:
                print(f"Skipped {path}: {e}")

    if news_frames:
        news_monthly = pd.concat(news_frames, ignore_index=True)
        wide = news_monthly.pivot_table(index="timestamp_utc", columns="source", values="tone", aggfunc="mean")
        wide.columns = [f"NEWS_{c}_TONE" for c in wide.columns]
        wide = wide.reset_index().sort_values("timestamp_utc")

        for col in ["NEWS_FED_TONE", "NEWS_ECB_TONE", "NEWS_BOE_TONE"]:
            if col not in wide.columns:
                wide[col] = 0.0

        wide["NEWS_ECB_MINUS_FED"] = wide["NEWS_ECB_TONE"] - wide["NEWS_FED_TONE"]
        wide["NEWS_BOE_MINUS_FED"] = wide["NEWS_BOE_TONE"] - wide["NEWS_FED_TONE"]
        news_monthly_wide = wide.sort_values("timestamp_utc").reset_index(drop=True)
    else:
        print("Warning: no readable news parquet files found; using empty news feature frame.")
        news_monthly_wide = pd.DataFrame(columns=["timestamp_utc"])

print(f"news_monthly_wide shape: {news_monthly_wide.shape}")
if len(news_monthly_wide) > 0:
    print("news date range:", news_monthly_wide["timestamp_utc"].min(), "->", news_monthly_wide["timestamp_utc"].max())
print("news columns:", news_monthly_wide.columns.tolist())

news_monthly_wide shape: (313, 6)
news date range: 2000-01-31 00:00:00+00:00 -> 2026-04-30 00:00:00+00:00
news columns: ['timestamp_utc', 'NEWS_BOE_TONE', 'NEWS_ECB_TONE', 'NEWS_FED_TONE', 'NEWS_ECB_MINUS_FED', 'NEWS_BOE_MINUS_FED']


In [6]:
ohlcv_by_pair: dict[str, pd.DataFrame] = {}
ohlcv_file_used: dict[str, Path] = {}
available_pairs: list[str] = []


def _pick_best_ohlcv_d1_file(pair: str, folder: Path) -> Path | None:
    candidates = sorted(folder.glob(f"ohlcv_{pair}*_D1_*.parquet"))
    if not candidates:
        return None

    scored = []
    for p in candidates:
        try:
            sdf = pd.read_parquet(p, columns=["timestamp_utc"])
            ts = pd.to_datetime(sdf["timestamp_utc"], utc=True, errors="coerce").dropna()
            if ts.empty:
                continue
            span_days = (ts.max() - ts.min()).days
            rows = len(ts)
            is_m_suffix = p.stem.endswith("m")
            scored.append((span_days, rows, 0 if is_m_suffix else 1, p))
        except Exception:
            continue

    if not scored:
        return None

    scored.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    return scored[0][3]

for pair in TRAINING_PAIRS:
    best_file = _pick_best_ohlcv_d1_file(pair, OHLCV_DIR)
    if best_file is None:
        print(f"{pair}: no readable D1 parquet found in {OHLCV_DIR}")
        continue

    pdf = pd.read_parquet(best_file)
    ts_col = "timestamp_utc" if "timestamp_utc" in pdf.columns else "timestamp"
    pdf["timestamp_utc"] = pd.to_datetime(pdf[ts_col], utc=True, errors="coerce")
    pdf = pdf.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)
    ohlcv_by_pair[pair] = pdf
    ohlcv_file_used[pair] = best_file
    available_pairs.append(pair)
    print(
        f"{pair}: file={best_file.name}, rows={len(pdf):,}, "
        f"range={pdf['timestamp_utc'].min()} -> {pdf['timestamp_utc'].max()}"
    )

print("Available pairs for training:", available_pairs)

EURUSD: file=ohlcv_EURUSD_D1_2003-05-04_2025-12-31.parquet, rows=7,282, range=2003-05-04 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
GBPUSD: file=ohlcv_GBPUSD_D1_2003-05-04_2025-12-31.parquet, rows=7,282, range=2003-05-04 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
USDJPY: file=ohlcv_USDJPY_D1_2003-05-04_2025-12-31.parquet, rows=7,045, range=2003-05-04 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
AUDUSD: file=ohlcv_AUDUSDm_D1_2021-01-03_2025-12-30.parquet, rows=1,560, range=2021-01-03 00:00:00+00:00 -> 2025-12-30 00:00:00+00:00
USDCHF: file=ohlcv_USDCHFm_D1_2021-01-03_2025-12-30.parquet, rows=1,560, range=2021-01-03 00:00:00+00:00 -> 2025-12-30 00:00:00+00:00
Available pairs for training: ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'USDCHF']


In [7]:
base = macro_monthly.copy().sort_values("timestamp_utc").reset_index(drop=True)
derived = pd.DataFrame({"timestamp_utc": base["timestamp_utc"]})
skipped_reasons: list[str] = []

available_series = set(c for c in base.columns if c != "timestamp_utc")


def _need(cols: list[str], feature_name: str) -> bool:
    missing = [c for c in cols if c not in available_series]
    if missing:
        skipped_reasons.append(f"{feature_name}: missing {missing}")
        return False
    return True

for s in ["DFF", "ECB_DFR", "ECB_MRR", "BOEBANKRATE"]:
    if s in available_series:
        derived[f"{s}_LVL"] = base[s]
        derived[f"{s}_LVL_L3"] = base[s].shift(3)
        derived[f"{s}_LVL_L6"] = base[s].shift(6)
    else:
        skipped_reasons.append(f"{s}_levels: missing [{s}]")

for s in ["DFF", "ECB_DFR", "BOEBANKRATE"]:
    if s in available_series:
        d1 = base[s].diff(1)
        d3 = base[s].diff(3)
        d6 = base[s].diff(6)
        derived[f"{s}_MOM_1M"] = d1
        derived[f"{s}_MOM_3M"] = d3
        derived[f"{s}_MOM_6M"] = d6
        derived[f"{s}_ACC"] = d1 - d3 / 3.0
    else:
        skipped_reasons.append(f"{s}_momentum: missing [{s}]")

if _need(["ECB_DFR", "DFF"], "RATEDIFF_EUR_USD"):
    rd = base["ECB_DFR"] - base["DFF"]
    derived["RATEDIFF_EUR_USD"] = rd
    derived["RATEDIFF_EUR_USD_L1"] = rd.shift(1)
    derived["RATEDIFF_EUR_USD_L3"] = rd.shift(3)
    derived["RATEDIFF_EUR_USD_L6"] = rd.shift(6)
    derived["RATEDIFF_EUR_USD_D3"] = rd.diff(3)
    derived["RATEDIFF_EUR_USD_D6"] = rd.diff(6)

if _need(["BOEBANKRATE", "DFF"], "RATEDIFF_GBP_USD"):
    derived["RATEDIFF_GBP_USD"] = base["BOEBANKRATE"] - base["DFF"]

if _need(["DFF", "CPIAUCSL"], "REALRATE_USD"):
    cpi_us_yoy = base["CPIAUCSL"].pct_change(12) * 100.0
    derived["CPI_US_YOY"] = cpi_us_yoy
    derived["REALRATE_USD"] = base["DFF"] - cpi_us_yoy

if _need(["ECB_DFR", "HICP_EA"], "REALRATE_EUR"):
    cpi_eur_yoy = base["HICP_EA"].pct_change(12) * 100.0
    derived["CPI_EUR_YOY"] = cpi_eur_yoy
    derived["REALRATE_EUR"] = base["ECB_DFR"] - cpi_eur_yoy

inflation_candidates = [c for c in available_series if ("CPI" in c or "PCE" in c)]
for s in sorted(inflation_candidates):
    derived[f"{s}_YOY"] = base[s].pct_change(12) * 100.0
if not inflation_candidates:
    skipped_reasons.append("Inflation_YOY: no CPI/PCE series available")

for s in ["UNRATE", "BOPGSTB", "NETFI", "PAYEMS", "GDPC1"]:
    if s in available_series:
        derived[f"{s}_YOY"] = base[s].pct_change(12) * 100.0
        derived[f"{s}_D1"] = base[s].diff(1)
    else:
        skipped_reasons.append(f"Activity_{s}: missing [{s}]")

for s in ["STLFSI4", "VIXCLS", "BAMLH0A0HYM2"]:
    if s in available_series:
        derived[f"{s}_LVL"] = base[s]
        derived[f"{s}_L1"] = base[s].shift(1)
        derived[f"{s}_L3"] = base[s].shift(3)
    else:
        skipped_reasons.append(f"Risk_{s}: missing [{s}]")

derived_feature_cols = [c for c in derived.columns if c != "timestamp_utc"]
print(f"Total derived features computed: {len(derived_feature_cols)}")
print("First 30 derived feature names:")
print(derived_feature_cols[:30])
print(f"Total skipped items: {len(skipped_reasons)}")
print("Skipped due to missing source series:")
for item in skipped_reasons:
    print(" -", item)

Total derived features computed: 41


First 30 derived feature names:
['DFF_LVL', 'DFF_LVL_L3', 'DFF_LVL_L6', 'ECB_DFR_LVL', 'ECB_DFR_LVL_L3', 'ECB_DFR_LVL_L6', 'ECB_MRR_LVL', 'ECB_MRR_LVL_L3', 'ECB_MRR_LVL_L6', 'DFF_MOM_1M', 'DFF_MOM_3M', 'DFF_MOM_6M', 'DFF_ACC', 'ECB_DFR_MOM_1M', 'ECB_DFR_MOM_3M', 'ECB_DFR_MOM_6M', 'ECB_DFR_ACC', 'RATEDIFF_EUR_USD', 'RATEDIFF_EUR_USD_L1', 'RATEDIFF_EUR_USD_L3', 'RATEDIFF_EUR_USD_L6', 'RATEDIFF_EUR_USD_D3', 'RATEDIFF_EUR_USD_D6', 'CPI_US_YOY', 'REALRATE_USD', 'CPIAUCSL_YOY', 'UNRATE_YOY', 'UNRATE_D1', 'BOPGSTB_YOY', 'BOPGSTB_D1']
Total skipped items: 6
Skipped due to missing source series:
 - BOEBANKRATE_levels: missing [BOEBANKRATE]
 - BOEBANKRATE_momentum: missing [BOEBANKRATE]
 - RATEDIFF_GBP_USD: missing ['BOEBANKRATE']
 - REALRATE_EUR: missing ['HICP_EA']
 - Activity_PAYEMS: missing [PAYEMS]
 - Activity_GDPC1: missing [GDPC1]


In [8]:
expected_news_cols = [
    "NEWS_FED_TONE",
    "NEWS_ECB_TONE",
    "NEWS_BOE_TONE",
    "NEWS_ECB_MINUS_FED",
    "NEWS_BOE_MINUS_FED",
]

if "timestamp_utc" in news_monthly_wide.columns and len(news_monthly_wide) > 0:
    feature_matrix = derived.merge(news_monthly_wide, on="timestamp_utc", how="left")
else:
    feature_matrix = derived.copy()

for c in expected_news_cols:
    if c not in feature_matrix.columns:
        feature_matrix[c] = 0.0

feature_matrix = feature_matrix.sort_values("timestamp_utc").reset_index(drop=True)
print(f"feature_matrix shape: {feature_matrix.shape}")
print("feature_matrix last 10 columns:")
print(feature_matrix.columns.tolist()[-10:])

feature_matrix shape: (62, 47)
feature_matrix last 10 columns:
['VIXCLS_L1', 'VIXCLS_L3', 'BAMLH0A0HYM2_LVL', 'BAMLH0A0HYM2_L1', 'BAMLH0A0HYM2_L3', 'NEWS_BOE_TONE', 'NEWS_ECB_TONE', 'NEWS_FED_TONE', 'NEWS_ECB_MINUS_FED', 'NEWS_BOE_MINUS_FED']


In [9]:
pair_datasets: dict[str, pd.DataFrame] = {}
pair_feature_cols: dict[str, list[str]] = {}

all_feature_cols = [c for c in feature_matrix.columns if c != "timestamp_utc"]

for pair in TRAINING_PAIRS:
    if pair not in ohlcv_by_pair:
        print(f"{pair}: skipped (missing OHLCV)")
        continue

    px = ohlcv_by_pair[pair].copy()
    close_col = "close" if "close" in px.columns else None
    if close_col is None:
        numeric_candidates = [
            c for c in px.columns if c.lower() in {"adj_close", "last", "price", "close_price"}
        ]
        if numeric_candidates:
            close_col = numeric_candidates[0]
        else:
            numeric_cols = [c for c in px.columns if pd.api.types.is_numeric_dtype(px[c])]
            close_col = numeric_cols[0]

    monthly_close = (
        px.assign(month_end=px["timestamp_utc"].dt.to_period("M").dt.to_timestamp("M").dt.tz_localize("UTC"))
        .groupby("month_end")[close_col]
        .last()
        .rename("monthly_close")
        .reset_index()
        .rename(columns={"month_end": "timestamp_utc"})
        .sort_values("timestamp_utc")
    )

    monthly_close["log_return"] = np.log(monthly_close["monthly_close"].shift(-1) / monthly_close["monthly_close"])

    if pair == "EURUSD":
        selected = [
            c
            for c in all_feature_cols
            if (
                "EUR" in c
                or "ECB" in c
                or c.startswith("DFF")
                or c.startswith("REALRATE_USD")
                or c.startswith("CPI_US")
                or c.startswith("STLFSI4")
                or c.startswith("VIX")
                or c.startswith("BAML")
                or c.startswith("NEWS_")
            )
        ]
    elif pair == "GBPUSD":
        selected = [
            c
            for c in all_feature_cols
            if (
                "GBP" in c
                or "BOE" in c
                or c.startswith("DFF")
                or c.startswith("REALRATE_USD")
                or c.startswith("CPI_US")
                or c.startswith("STLFSI4")
                or c.startswith("VIX")
                or c.startswith("BAML")
                or c.startswith("NEWS_")
            )
        ]
    else:
        selected = [
            c
            for c in all_feature_cols
            if (
                c.startswith("DFF")
                or c.startswith("REALRATE_USD")
                or c.startswith("CPI_US")
                or c.startswith("RATEDIFF_")
                or c.startswith("STLFSI4")
                or c.startswith("VIX")
                or c.startswith("BAML")
                or c.startswith("NEWS_")
            )
        ]

    selected = sorted(set(selected))
    pair_feature_cols[pair] = selected

    df_pair = feature_matrix[["timestamp_utc", *selected]].merge(
        monthly_close[["timestamp_utc", "monthly_close", "log_return"]],
        on="timestamp_utc",
        how="inner",
    )
    df_pair["pair"] = pair
    df_pair = df_pair.dropna(subset=["log_return"]).sort_values("timestamp_utc").reset_index(drop=True)

    pos_rate = float((df_pair["log_return"] > 0).mean()) if len(df_pair) else np.nan
    pair_datasets[pair] = df_pair

    print(
        f"{pair}: rows={len(df_pair)}, features={len(selected)}, "
        f"positive_rate={pos_rate:.3f}"
    )

EURUSD: rows=59, features=39, positive_rate=0.525


GBPUSD: rows=59, features=23, positive_rate=0.525
USDJPY: rows=59, features=29, positive_rate=0.593
AUDUSD: rows=59, features=29, positive_rate=0.525
USDCHF: rows=59, features=29, positive_rate=0.441


In [10]:
def kalman_interpolate(series: np.ndarray) -> np.ndarray:
    x = np.asarray(series, dtype=float).copy()
    n = len(x)
    if n == 0:
        return x

    dt = 1.0
    F = np.array([[1.0, dt], [0.0, 1.0]])
    H = np.array([[1.0, 0.0]])
    Q = np.array([[1e-4, 0.0], [0.0, 1e-3]])
    R = np.array([[1e-2]])

    valid = np.isfinite(x)
    if not valid.any():
        return np.zeros_like(x)

    first = np.flatnonzero(valid)[0]
    state = np.array([x[first], 0.0])
    P = np.eye(2)

    pred_states = np.zeros((n, 2))
    pred_covs = np.zeros((n, 2, 2))
    filt_states = np.zeros((n, 2))
    filt_covs = np.zeros((n, 2, 2))

    for t in range(n):
        state = F @ state
        P = F @ P @ F.T + Q
        pred_states[t], pred_covs[t] = state, P

        if np.isfinite(x[t]):
            y = np.array([x[t]]) - H @ state
            S = H @ P @ H.T + R
            K = P @ H.T @ np.linalg.inv(S)
            state = state + (K @ y).ravel()
            P = (np.eye(2) - K @ H) @ P

        filt_states[t], filt_covs[t] = state, P

    smooth_states = filt_states.copy()
    smooth_covs = filt_covs.copy()

    for t in range(n - 2, -1, -1):
        C = filt_covs[t] @ F.T @ np.linalg.inv(pred_covs[t + 1])
        smooth_states[t] = filt_states[t] + C @ (smooth_states[t + 1] - pred_states[t + 1])
        smooth_covs[t] = filt_covs[t] + C @ (smooth_covs[t + 1] - pred_covs[t + 1]) @ C.T

    out = x.copy()
    missing = ~np.isfinite(out)
    out[missing] = smooth_states[missing, 0]
    return out

synthetic = np.array([1.0, 1.1, np.nan, 1.4, 1.6, np.nan, 1.9])
filled = kalman_interpolate(synthetic)
print("Synthetic before:", synthetic)
print("Synthetic after :", np.round(filled, 4))

Synthetic before: [1.  1.1 nan 1.4 1.6 nan 1.9]
Synthetic after : [1.     1.1    1.2727 1.4    1.6    1.7382 1.9   ]


In [11]:
from src.ingestion.preprocessors.macro_signal_builder import MacroHybridArtifact

@dataclass
class PairResult:
    pair: str
    test_timestamps: list
    y_test: np.ndarray
    y_prev_test: np.ndarray
    hybrid_predictions: np.ndarray
    metrics: dict


class ResidualLSTM(nn.Module):
    def __init__(self, hidden_size: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def train_hybrid_var_lstm(pair: str, df_pair: pd.DataFrame) -> tuple[PairResult, MacroHybridArtifact]:
    df = df_pair.sort_values("timestamp_utc").reset_index(drop=True).copy()

    feature_cols = [
        c
        for c in df.columns
        if c not in {"timestamp_utc", "pair", "monthly_close", "log_return", "target", "target_next"}
    ]
    if len(feature_cols) == 0:
        raise ValueError(f"{pair}: no feature columns")

    X_raw = df[feature_cols].copy()
    for c in feature_cols:
        X_raw[c] = kalman_interpolate(pd.to_numeric(X_raw[c], errors="coerce").to_numpy(dtype=float))

    y = pd.to_numeric(df["log_return"], errors="coerce").to_numpy(dtype=float)
    ts = pd.to_datetime(df["timestamp_utc"], utc=True)

    n = len(df)
    split_idx = int(np.floor(n * TRAIN_RATIO))
    split_idx = min(max(split_idx, 24), n - 6)
    if split_idx <= LOOKBACK_MONTHS + 4:
        raise ValueError(f"{pair}: not enough samples after split, n={n}, split_idx={split_idx}")

    X_train_raw = X_raw.iloc[:split_idx].copy()
    X_test_raw = X_raw.iloc[split_idx:].copy()
    y_train = y[:split_idx]
    y_test = y[split_idx:]
    test_timestamps = ts.iloc[split_idx:].tolist()
    y_prev_test = pd.Series(y).shift(1).iloc[split_idx:].fillna(0.0).to_numpy(dtype=float)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    train_var = np.nanvar(X_train, axis=0)
    finite_idx = np.where(np.isfinite(train_var))[0]
    ranked = finite_idx[np.argsort(train_var[finite_idx])[::-1]]
    topk = ranked[: min(VAR_MAX_FEATS, len(ranked))]

    var_cols = ["y"] + [feature_cols[i] for i in topk]
    var_train_df = pd.DataFrame(
        np.column_stack([y_train, X_train[:, topk]]),
        columns=var_cols,
    ).replace([np.inf, -np.inf], np.nan).dropna()

    if len(var_train_df) < 20:
        raise ValueError(f"{pair}: insufficient finite rows for VAR after cleaning")

    max_lag = max(1, min(4, len(var_train_df) // 6))
    selected_aic = []
    for lag in range(1, max_lag + 1):
        try:
            fit = VAR(var_train_df).fit(lag)
            selected_aic.append((lag, fit.aic))
        except Exception:
            continue

    if not selected_aic:
        raise RuntimeError(f"{pair}: VAR lag selection failed")

    best_lag = min(selected_aic, key=lambda x: x[1])[0]
    var_fit = VAR(var_train_df).fit(best_lag)

    history = var_train_df.values.copy()
    var_forecast = []
    for i in range(len(y_test)):
        exo_row = X_test[i, topk] if len(topk) > 0 else np.array([])
        yhat_full = var_fit.forecast(y=history[-best_lag:], steps=1)[0]
        yhat = float(yhat_full[0])
        var_forecast.append(yhat)
        next_state = np.concatenate([[yhat], exo_row]) if len(exo_row) > 0 else np.array([yhat])
        history = np.vstack([history, next_state])

    var_forecast = np.asarray(var_forecast, dtype=float)

    fitted_vals = var_fit.fittedvalues["y"].to_numpy(dtype=float)
    resid_train = y_train[best_lag : best_lag + len(fitted_vals)] - fitted_vals
    resid_train = resid_train[np.isfinite(resid_train)]

    if len(resid_train) <= LOOKBACK_MONTHS + 8:
        raise ValueError(f"{pair}: insufficient residual samples for LSTM")

    seq_x = []
    seq_y = []
    for i in range(LOOKBACK_MONTHS, len(resid_train)):
        seq_x.append(resid_train[i - LOOKBACK_MONTHS : i])
        seq_y.append(resid_train[i])

    seq_x = np.asarray(seq_x, dtype=np.float32)
    seq_y = np.asarray(seq_y, dtype=np.float32)

    n_seq = len(seq_x)
    val_n = max(1, int(0.2 * n_seq))
    tr_n = n_seq - val_n

    Xtr = torch.tensor(seq_x[:tr_n], dtype=torch.float32).unsqueeze(-1)
    ytr = torch.tensor(seq_y[:tr_n], dtype=torch.float32).unsqueeze(-1)
    Xva = torch.tensor(seq_x[tr_n:], dtype=torch.float32).unsqueeze(-1)
    yva = torch.tensor(seq_y[tr_n:], dtype=torch.float32).unsqueeze(-1)

    model = ResidualLSTM(hidden_size=32)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    best_state = None
    best_val = np.inf
    patience = 10
    no_improve = 0

    for _ in range(250):
        model.train()
        opt.zero_grad()
        pred = model(Xtr)
        loss = loss_fn(pred, ytr)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(Xva)
            val_loss = float(loss_fn(val_pred, yva).item())

        if val_loss < best_val - 1e-8:
            best_val = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    rolling = resid_train[-LOOKBACK_MONTHS:].astype(np.float32).copy()
    lstm_resid_forecast = []
    with torch.no_grad():
        for _ in range(len(y_test)):
            xin = torch.tensor(rolling, dtype=torch.float32).view(1, LOOKBACK_MONTHS, 1)
            pred_resid = float(model(xin).item())
            lstm_resid_forecast.append(pred_resid)
            rolling = np.concatenate([rolling[1:], np.array([pred_resid], dtype=np.float32)])

    lstm_resid_forecast = np.asarray(lstm_resid_forecast, dtype=float)

    hybrid_predictions = var_forecast + lstm_resid_forecast
    hybrid_predictions = hybrid_predictions[: len(y_test)]
    if len(hybrid_predictions) != len(y_test):
        raise RuntimeError(f"{pair}: prediction length mismatch")

    rmse = float(np.sqrt(mean_squared_error(y_test, hybrid_predictions)))
    mae = float(mean_absolute_error(y_test, hybrid_predictions))
    y_true_dir = (y_test > y_prev_test).astype(int)
    y_pred_dir = (hybrid_predictions > y_prev_test).astype(int)
    directional_acc = float((y_true_dir == y_pred_dir).mean())
    balanced_acc = float(balanced_accuracy_score(y_true_dir, y_pred_dir))
    score = hybrid_predictions - y_prev_test
    try:
        roc_auc = float(roc_auc_score(y_true_dir, score)) if len(np.unique(y_true_dir)) > 1 else np.nan
    except Exception:
        roc_auc = np.nan

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "DirectionalAccuracy": directional_acc,
        "BalancedAccuracy": balanced_acc,
        "ROC_AUC": roc_auc,
        "best_lag": int(best_lag),
        "n_features": int(len(feature_cols)),
        "var_features": [feature_cols[i] for i in topk],
    }

    result = PairResult(
        pair=pair,
        test_timestamps=test_timestamps,
        y_test=y_test,
        y_prev_test=y_prev_test,
        hybrid_predictions=hybrid_predictions,
        metrics=metrics,
    )

    artifact = MacroHybridArtifact(
        pair=pair,
        feature_cols=feature_cols,
        var_top_indices=topk.tolist(),
        best_lag=int(best_lag),
        var_fit=var_fit,
        lstm_state_dict=best_state or {},
        scaler=scaler,
        var_history_tail=history[-best_lag:].copy(),
        resid_tail=rolling.copy(),
        train_cutoff=ts.iloc[split_idx - 1],
    )

    return result, artifact


print("Defined PairResult, ResidualLSTM, MacroHybridArtifact, and train_hybrid_var_lstm")

Defined PairResult, ResidualLSTM, MacroHybridArtifact, and train_hybrid_var_lstm


In [12]:
pair_results: list[PairResult] = []
pair_artifacts: list[MacroHybridArtifact] = []
training_errors: dict[str, str] = {}

for pair in TRAINING_PAIRS:
    df_pair = pair_datasets.get(pair)
    if df_pair is None or len(df_pair) == 0:
        training_errors[pair] = "missing or empty dataset"
        print(f"{pair}: skipped -> missing or empty dataset")
        continue

    try:
        result, artifact = train_hybrid_var_lstm(pair, df_pair)
        pair_results.append(result)
        pair_artifacts.append(artifact)
        ts_min = min(result.test_timestamps)
        ts_max = max(result.test_timestamps)
        print(
            f"{pair}: test_range={ts_min.date()} -> {ts_max.date()}, "
            f"n_test={len(result.y_test)}, RMSE={result.metrics['RMSE']:.6f}, "
            f"ROC_AUC={result.metrics['ROC_AUC']:.4f}, "
            f"DirAcc={result.metrics['DirectionalAccuracy']:.4f}"
        )
    except Exception as e:
        training_errors[pair] = str(e)
        print(f"{pair}: failed -> {e}")

print("Completed pairs:", [r.pair for r in pair_results])
print("Errors:", training_errors)

EURUSD: test_range=2024-12-31 -> 2025-11-30, n_test=12, RMSE=0.031753, ROC_AUC=0.7188, DirAcc=0.5833


GBPUSD: test_range=2024-12-31 -> 2025-11-30, n_test=12, RMSE=0.043622, ROC_AUC=0.8438, DirAcc=0.5000


USDJPY: test_range=2024-12-31 -> 2025-11-30, n_test=12, RMSE=0.096116, ROC_AUC=0.3714, DirAcc=0.4167


AUDUSD: test_range=2024-12-31 -> 2025-11-30, n_test=12, RMSE=0.056126, ROC_AUC=0.7429, DirAcc=0.5833


USDCHF: test_range=2024-12-31 -> 2025-11-30, n_test=12, RMSE=0.036519, ROC_AUC=0.6562, DirAcc=0.4167
Completed pairs: ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'USDCHF']
Errors: {}


In [13]:
def rolling_zscore(s: pd.Series, window: int = 24) -> pd.Series:
    mu = s.rolling(window=window, min_periods=6).mean()
    sd = s.rolling(window=window, min_periods=6).std(ddof=0)
    z = (s - mu) / sd.replace(0.0, np.nan)
    return z.replace([np.inf, -np.inf], np.nan)


feature_ts = feature_matrix.set_index("timestamp_utc").sort_index()
signal_rows = []

for res in pair_results:
    test_idx = pd.DatetimeIndex(pd.to_datetime(res.test_timestamps, utc=True))
    score = pd.Series(res.hybrid_predictions - res.y_prev_test, index=test_idx)

    roll_max = score.abs().rolling(window=24, min_periods=1).max().replace(0.0, np.nan)
    macro_conf = (score.abs() / roll_max).clip(0.0, 1.0).fillna(0.0)

    if res.pair == "EURUSD" and "RATEDIFF_EUR_USD" in feature_ts.columns:
        carry_raw = feature_ts["RATEDIFF_EUR_USD"]
    elif res.pair == "GBPUSD" and "RATEDIFF_GBP_USD" in feature_ts.columns:
        carry_raw = feature_ts["RATEDIFF_GBP_USD"]
    elif "RATEDIFF_EUR_USD" in feature_ts.columns:
        carry_raw = feature_ts["RATEDIFF_EUR_USD"]
    else:
        carry_raw = pd.Series(0.0, index=feature_ts.index)

    carry_z = rolling_zscore(carry_raw, window=24).reindex(test_idx).fillna(0.0)

    dff_vel = feature_ts["DFF_MOM_3M"] if "DFF_MOM_3M" in feature_ts.columns else pd.Series(0.0, index=feature_ts.index)
    stlfsi = feature_ts["STLFSI4_LVL"] if "STLFSI4_LVL" in feature_ts.columns else pd.Series(0.0, index=feature_ts.index)
    regime = 0.5 * (
        rolling_zscore(dff_vel, window=24).reindex(test_idx).fillna(0.0)
        + rolling_zscore(stlfsi, window=24).reindex(test_idx).fillna(0.0)
    )

    real_usd = feature_ts["REALRATE_USD"] if "REALRATE_USD" in feature_ts.columns else pd.Series(0.0, index=feature_ts.index)
    real_dev = real_usd - real_usd.rolling(window=24, min_periods=6).mean()
    fundamental = rolling_zscore(real_dev, window=24).reindex(test_idx).fillna(0.0)

    direction = np.where(score.values >= 0.0, "up", "down")
    macro_surprise = np.zeros(len(test_idx), dtype=float)
    macro_bias = np.clip(
        0.5 * macro_conf.values + 0.25 * np.abs(carry_z.values) + 0.25 * np.abs(fundamental.values),
        0.0,
        1.0,
    )

    tmp = pd.DataFrame(
        {
            "timestamp_utc": test_idx,
            "pair": res.pair,
            "module_c_direction": direction,
            "macro_confidence": macro_conf.values,
            "carry_signal_score": carry_z.values,
            "regime_context_score": regime.values,
            "fundamental_mispricing_score": fundamental.values,
            "macro_surprise_score": macro_surprise,
            "macro_bias_score": macro_bias,
        }
    )
    signal_rows.append(tmp)

if signal_rows:
    macro_signal_df = pd.concat(signal_rows, ignore_index=True).sort_values(["pair", "timestamp_utc"]).reset_index(drop=True)
else:
    macro_signal_df = pd.DataFrame(
        columns=[
            "timestamp_utc",
            "pair",
            "module_c_direction",
            "macro_confidence",
            "carry_signal_score",
            "regime_context_score",
            "fundamental_mispricing_score",
            "macro_surprise_score",
            "macro_bias_score",
        ]
    )

print("macro_signal_df shape:", macro_signal_df.shape)
print("pair distribution:")
print(macro_signal_df["pair"].value_counts(dropna=False))
print("direction balance per pair:")
print(
    macro_signal_df.groupby("pair")["module_c_direction"]
    .value_counts(normalize=True)
    .rename("share")
)

num_cols = [
    "macro_confidence",
    "carry_signal_score",
    "regime_context_score",
    "fundamental_mispricing_score",
    "macro_surprise_score",
    "macro_bias_score",
]
print("numeric describe:")
print(macro_signal_df[num_cols].describe().T)

macro_signal_df shape: (60, 9)
pair distribution:
pair
AUDUSD    12
EURUSD    12
GBPUSD    12
USDCHF    12
USDJPY    12
Name: count, dtype: int64
direction balance per pair:
pair    module_c_direction
AUDUSD  down                  0.666667
        up                    0.333333
EURUSD  up                    0.583333
        down                  0.416667
GBPUSD  down                  0.833333
        up                    0.166667
USDCHF  up                    0.583333
        down                  0.416667
USDJPY  down                  0.666667
        up                    0.333333
Name: share, dtype: float64
numeric describe:
                              count      mean       std       min       25%  \
macro_confidence               60.0  0.557894  0.347758  0.024027  0.249408   
carry_signal_score             60.0 -1.211574  1.219983 -2.654954 -2.297990   
regime_context_score           60.0 -0.069666  0.514680 -1.119196 -0.225573   
fundamental_mispricing_score   60.0 -1.434712  

In [14]:
for res in pair_results:
    ts = pd.to_datetime(res.test_timestamps, utc=True)
    assert len(ts) > 0, f"{res.pair}: empty test_timestamps"
    assert ts.notna().all(), f"{res.pair}: null in test_timestamps"
    assert ts.is_monotonic_increasing, f"{res.pair}: test_timestamps not sorted"
    assert len(res.test_timestamps) == len(res.hybrid_predictions), f"{res.pair}: length mismatch"

required_contract_cols = [
    "module_c_direction",
    "macro_confidence",
    "carry_signal_score",
    "regime_context_score",
    "fundamental_mispricing_score",
    "macro_surprise_score",
    "macro_bias_score",
]
for c in required_contract_cols:
    assert c in macro_signal_df.columns, f"Missing contract field: {c}"

assert macro_signal_df["module_c_direction"].notna().all(), "NaN direction found"
assert macro_signal_df["macro_confidence"].notna().all(), "NaN confidence found"

print("Validation checks passed")
print("Contract sample: last 3 rows per pair")
sample = (
    macro_signal_df.sort_values(["pair", "timestamp_utc"])
    .groupby("pair", as_index=False)
    .tail(3)
    .sort_values(["pair", "timestamp_utc"])
)
print(sample)

Validation checks passed
Contract sample: last 3 rows per pair
               timestamp_utc    pair module_c_direction  macro_confidence  \
9  2025-09-30 00:00:00+00:00  AUDUSD               down          0.299149   
10 2025-10-31 00:00:00+00:00  AUDUSD               down          0.826844   
11 2025-11-30 00:00:00+00:00  AUDUSD               down          0.415510   
21 2025-09-30 00:00:00+00:00  EURUSD                 up          0.083499   
22 2025-10-31 00:00:00+00:00  EURUSD                 up          0.362729   
23 2025-11-30 00:00:00+00:00  EURUSD               down          0.341090   
33 2025-09-30 00:00:00+00:00  GBPUSD               down          0.293513   
34 2025-10-31 00:00:00+00:00  GBPUSD               down          0.071050   
35 2025-11-30 00:00:00+00:00  GBPUSD               down          0.264827   
45 2025-09-30 00:00:00+00:00  USDCHF               down          0.150714   
46 2025-10-31 00:00:00+00:00  USDCHF                 up          0.081495   
47 2025-11-30

In [15]:
models_out = ROOT / "models" / "production" / "macro"
macro_out = ROOT / "data" / "processed" / "macro"
models_out.mkdir(parents=True, exist_ok=True)
macro_out.mkdir(parents=True, exist_ok=True)

# Legacy runs pickle (historical PairResult objects)
pickle_path = models_out / "hybrid_var_lstm_runs.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(pair_results, f)
print(f"Saved runs pickle: {pickle_path} | pairs={len(pair_results)}")

# Inference artifacts (VAR fit + LSTM weights + scaler — used by MacroSignalBuilder.run())
artifacts_path = models_out / "macro_hybrid_models.pkl"
with open(artifacts_path, "wb") as f:
    pickle.dump(pair_artifacts, f)
with open(artifacts_path, "rb") as f:
    loaded_artifacts = pickle.load(f)
assert len(loaded_artifacts) == len(pair_artifacts), "Artifact count mismatch"
print(f"Saved inference artifacts: {artifacts_path} | pairs={len(loaded_artifacts)}")
for a in loaded_artifacts:
    print(f"  {a.pair}: features={len(a.feature_cols)}, best_lag={a.best_lag}, train_cutoff={a.train_cutoff.date()}")

# macro_signal.parquet
parquet_path = macro_out / "macro_signal.parquet"
macro_signal_df.to_parquet(parquet_path, engine="pyarrow", index=False)
loaded_signal = pd.read_parquet(parquet_path)
assert len(loaded_signal) == len(macro_signal_df), "Parquet row count mismatch"
print(f"Saved parquet: {parquet_path} | shape={loaded_signal.shape}")
print(loaded_signal.dtypes)

summary_rows = []
for r in pair_results:
    ts = pd.to_datetime(r.test_timestamps, utc=True)
    summary_rows.append(
        {
            "pair": r.pair,
            "test_start": ts.min().date(),
            "test_end": ts.max().date(),
            "n_months": len(ts),
            "roc_auc": r.metrics.get("ROC_AUC", np.nan),
            "directional_accuracy": r.metrics.get("DirectionalAccuracy", np.nan),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("pair").reset_index(drop=True)
print("Final summary:")
print(summary_df)

Saved runs pickle: D:\SCRIPTS\FX-AlphaLab\models\production\macro\hybrid_var_lstm_runs.pkl | pairs=5
Saved inference artifacts: D:\SCRIPTS\FX-AlphaLab\models\production\macro\macro_hybrid_models.pkl | pairs=5
  EURUSD: features=39, best_lag=4, train_cutoff=2024-11-30
  GBPUSD: features=23, best_lag=3, train_cutoff=2024-11-30
  USDJPY: features=29, best_lag=4, train_cutoff=2024-11-30
  AUDUSD: features=29, best_lag=4, train_cutoff=2024-11-30
  USDCHF: features=29, best_lag=4, train_cutoff=2024-11-30
Saved parquet: D:\SCRIPTS\FX-AlphaLab\data\processed\macro\macro_signal.parquet | shape=(60, 9)
timestamp_utc                   datetime64[ns, UTC]
pair                                         object
module_c_direction                           object
macro_confidence                            float64
carry_signal_score                          float64
regime_context_score                        float64
fundamental_mispricing_score                float64
macro_surprise_score                

In [16]:
for p, d in pair_datasets.items():
    if len(d) == 0:
        continue
    print(p, d['timestamp_utc'].min(), d['timestamp_utc'].max(), len(d))

EURUSD 2021-01-31 00:00:00+00:00 2025-11-30 00:00:00+00:00 59
GBPUSD 2021-01-31 00:00:00+00:00 2025-11-30 00:00:00+00:00 59
USDJPY 2021-01-31 00:00:00+00:00 2025-11-30 00:00:00+00:00 59
AUDUSD 2021-01-31 00:00:00+00:00 2025-11-30 00:00:00+00:00 59
USDCHF 2021-01-31 00:00:00+00:00 2025-11-30 00:00:00+00:00 59
